# KPI Analysis Notebook

## Project

RetailOps Insight — Retail Operations Analytics, Data Quality, and Support Dashboard

## Purpose

This notebook reviews the KPI outputs generated from SQL analysis.

The goal is to interpret business performance, reporting logic, and data quality risk using Python and pandas.

## Questions Reviewed

- What are the main executive-level KPIs?
- What is the completed sales amount?
- What is the cancellation/return amount?
- What is the net sales amount after cancellations?
- What is the average order value?
- How do monthly KPIs change over time?
- What are the major data quality risks?
- Which countries and products contribute strongly to sales?

In [1]:
from pathlib import Path

import pandas as pd

In [2]:
current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

sql_output_dir = project_root / "reports" / "sql_outputs"

print("Current directory:", current_dir)
print("Project root:", project_root)
print("SQL output directory:", sql_output_dir)
print("SQL output directory exists:", sql_output_dir.exists())

Current directory: /home/tirtha/Projects/retailops-insight/notebooks
Project root: /home/tirtha/Projects/retailops-insight
SQL output directory: /home/tirtha/Projects/retailops-insight/reports/sql_outputs
SQL output directory exists: True


In [3]:
executive_kpi = pd.read_csv(sql_output_dir / "executive_kpi_summary.csv")
monthly_kpi = pd.read_csv(sql_output_dir / "monthly_kpi_summary.csv")
data_quality_kpi = pd.read_csv(sql_output_dir / "data_quality_kpi_summary.csv")

transaction_status = pd.read_csv(sql_output_dir / "transaction_status_summary.csv")
top_countries = pd.read_csv(sql_output_dir / "top_countries_by_sales.csv")
top_products = pd.read_csv(sql_output_dir / "top_products_by_sales.csv")
customer_coverage = pd.read_csv(sql_output_dir / "customer_id_coverage.csv")

print("KPI and SQL output files loaded successfully.")
print("Executive KPI shape:", executive_kpi.shape)
print("Monthly KPI shape:", monthly_kpi.shape)
print("Data quality KPI shape:", data_quality_kpi.shape)
print("Top countries shape:", top_countries.shape)
print("Top products shape:", top_products.shape)

KPI and SQL output files loaded successfully.
Executive KPI shape: (1, 18)
Monthly KPI shape: (13, 9)
Data quality KPI shape: (1, 9)
Top countries shape: (10, 3)
Top products shape: (10, 5)


## Executive KPI Summary

This section reviews the main project-level KPIs generated from the full cleaned transaction dataset.

In [5]:
executive_kpi

,total_transaction_lines,completed_transaction_lines,cancelled_return_lines,data_issue_lines,completed_invoice_count,total_completed_sales_amount,total_cancelled_amount,net_sales_after_cancellations,average_order_value,average_completed_line_value,cancellation_amount_rate_percent,unique_product_count,known_customer_count,country_count,missing_customer_id_rows,missing_customer_id_rate_percent,duplicate_rows,duplicate_row_rate_percent
0,541909,530104,10624,1181,19960,10666684.54,896812.49,9769872.05,534.4,20.12,7.76,4070,4372,38,135080,24.93,10147,1.87


In [6]:
executive_kpi_transposed = (
    executive_kpi
    .T
    .reset_index()
)

executive_kpi_transposed.columns = ["metric", "value"]

executive_kpi_transposed

,metric,value
0,total_transaction_lines,541909.00
1,completed_transaction_lines,530104.00
2,cancelled_return_lines,10624.00
3,data_issue_lines,1181.00
4,completed_invoice_count,19960.00
5,total_completed_sales_amount,10666684.54
6,total_cancelled_amount,896812.49
7,net_sales_after_cancellations,9769872.05
8,average_order_value,534.40
9,average_completed_line_value,20.12


In [11]:
kpi = executive_kpi.iloc[0]

completed_sales = kpi["total_completed_sales_amount"]
cancelled_amount = kpi["total_cancelled_amount"]
net_sales = kpi["net_sales_after_cancellations"]
average_order_value = kpi["average_order_value"]
cancellation_rate = kpi["cancellation_amount_rate_percent"]
missing_customer_rate = kpi["missing_customer_id_rate_percent"]
duplicate_rate = kpi["duplicate_row_rate_percent"]

print(f"Completed sales amount: ${completed_sales:,.2f}")
print(f"Cancelled amount: ${cancelled_amount:,.2f}")
print(f"Net sales after cancellations: ${net_sales:,.2f}")
print(f"Average order value: ${average_order_value:,.2f}")
print(f"Cancellation amount rate: {cancellation_rate:.2f}%")
print(f"Missing customer ID rate: {missing_customer_rate:.2f}%")
print(f"Duplicate row rate: {duplicate_rate:.2f}%")

Completed sales amount: $10,666,684.54
Cancelled amount: $896,812.49
Net sales after cancellations: $9,769,872.05
Average order value: $534.40
Cancellation amount rate: 7.76%
Missing customer ID rate: 24.93%
Duplicate row rate: 1.87%


## Executive Interpretation

The full cleaned dataset contains over 541K transaction lines.

The project reports approximately $10.67M in completed sales and approximately $896.8K in cancelled or returned transaction value.

After subtracting cancelled/return value from completed sales, simplified net sales are approximately $9.77M.

The average completed order value is approximately $534.40.

From a data quality perspective, missing customer IDs affect approximately 24.93% of transaction rows, which means customer-level reporting should be handled carefully.

Duplicate rows affect approximately 1.87% of transaction rows and should be monitored as a reporting risk.

## Monthly KPI Summary

This section reviews monthly completed sales, cancellations, net sales, average order value, and data issue rows.

In [12]:
monthly_kpi

,invoice_year_month,total_transaction_lines,completed_transaction_lines,completed_invoice_count,completed_sales_amount,cancelled_amount,net_sales_after_cancellations,average_order_value,data_issue_rows
0,2010-12,42481,41480,1559,823746.14,74789.12,748957.02,528.38,203
1,2011-01,35147,34306,1086,691364.56,131364.30,560000.26,636.62,44
2,2011-02,27707,27105,1100,523631.89,25569.24,498062.65,476.03,79
3,2011-03,36748,35803,1454,717639.36,34372.28,683267.08,493.56,112
4,2011-04,29916,29096,1246,537808.62,44601.50,493207.12,431.63,75
5,2011-05,37030,36164,1681,770536.02,47202.51,723333.51,458.38,128
6,2011-06,36874,35977,1533,761739.90,70616.78,691123.12,496.89,79
7,2011-07,39518,38645,1475,719221.19,37921.08,681300.11,487.61,71
8,2011-08,35284,34483,1361,759138.38,54333.75,704804.63,557.78,83
9,2011-09,50226,49261,1837,1058590.17,38902.55,1019687.62,576.26,62


In [21]:
top_sales_month = monthly_kpi.sort_values(
    by="completed_sales_amount",
    ascending=False
).head(1)


top_sales_transposed = (
    top_sales_month
    .T
    .reset_index()
)

top_sales_transposed.columns = ["metric", "value"]
#top_sales_month
top_sales_transposed

,metric,value
0,invoice_year_month,2011-11
1,total_transaction_lines,84711
2,completed_transaction_lines,83369
3,completed_invoice_count,2769
4,completed_sales_amount,1509496.33
5,cancelled_amount,47740.08
6,net_sales_after_cancellations,1461756.25
7,average_order_value,545.14
8,data_issue_rows,129


In [37]:
highest_cancellation_month = monthly_kpi.sort_values(
    by="cancelled_amount",
    ascending=False
).head(1)

highest_cancellation_month
#top_sales_month_transposed=(top_sales_month.T.reset_index())
#top_sales_month_transposed.columns=['metric', 'value']
#top_sales_month_transposed

,invoice_year_month,total_transaction_lines,completed_transaction_lines,completed_invoice_count,completed_sales_amount,cancelled_amount,net_sales_after_cancellations,average_order_value,data_issue_rows
12,2011-12,25525,25111,819,638792.68,205124.67,433668.01,779.97,24


In [32]:
highest_data_issue_month = monthly_kpi.sort_values(
    by="data_issue_rows",
    ascending=False
).head(1)

highest_data_issue_month

,invoice_year_month,total_transaction_lines,completed_transaction_lines,completed_invoice_count,completed_sales_amount,cancelled_amount,net_sales_after_cancellations,average_order_value,data_issue_rows
0,2010-12,42481,41480,1559,823746.14,74789.12,748957.02,528.38,203


In [34]:
highest_data_issue_month = monthly_kpi.sort_values(
    by="data_issue_rows",
    ascending=False
).head(1)

highest_data_issue_month

,invoice_year_month,total_transaction_lines,completed_transaction_lines,completed_invoice_count,completed_sales_amount,cancelled_amount,net_sales_after_cancellations,average_order_value,data_issue_rows
0,2010-12,42481,41480,1559,823746.14,74789.12,748957.02,528.38,203


In [38]:
monthly_findings = pd.DataFrame({
    "finding": [
        "Highest completed sales month",
        "Highest cancellation amount month",
        "Highest data issue row month"
    ],
    "invoice_year_month": [
        top_sales_month["invoice_year_month"].iloc[0],
        highest_cancellation_month["invoice_year_month"].iloc[0],
        highest_data_issue_month["invoice_year_month"].iloc[0]
    ],
    "value": [
        top_sales_month["completed_sales_amount"].iloc[0],
        highest_cancellation_month["cancelled_amount"].iloc[0],
        highest_data_issue_month["data_issue_rows"].iloc[0]
    ]
})

monthly_findings

,finding,invoice_year_month,value
0,Highest completed sales month,2011-12,638792.68
1,Highest cancellation amount month,2011-12,205124.67
2,Highest data issue row month,2010-12,203.00


## Monthly KPI Interpretation

Monthly KPI analysis helps identify changes in sales performance, cancellation value, and data quality risk over time.

The highest completed sales month can indicate peak sales activity.

The highest cancellation month may require operational review because cancellations and returns reduce confidence in gross sales activity.

The highest data issue month may point to source-system, data-entry, or reporting-process problems that require investigation.

In [39]:
data_quality_kpi

,total_transaction_lines,missing_customer_id_rows,missing_customer_id_rate_percent,duplicate_rows,duplicate_row_rate_percent,zero_or_negative_unit_price_rows,missing_description_rows,data_issue_rows,data_quality_issue_records
0,541909,135080,24.93,10147,1.87,2517,1454,1181,169110


In [40]:
data_quality_kpi_transposed = (
    data_quality_kpi
    .T
    .reset_index()
)

data_quality_kpi_transposed.columns = ["metric", "value"]

data_quality_kpi_transposed

,metric,value
0,total_transaction_lines,541909.00
1,missing_customer_id_rows,135080.00
2,missing_customer_id_rate_percent,24.93
3,duplicate_rows,10147.00
4,duplicate_row_rate_percent,1.87
5,zero_or_negative_unit_price_rows,2517.00
6,missing_description_rows,1454.00
7,data_issue_rows,1181.00
8,data_quality_issue_records,169110.00


## Customer ID Coverage

This section reviews how much completed sales is linked to known vs missing customer IDs.

In [41]:
customer_coverage

,customer_id_status,transaction_lines,completed_sales_amount
0,Known Customer ID,406829,8911407.90
1,Missing Customer ID,135080,1755276.64


In [42]:
customer_coverage["completed_sales_share_percent"] = (
    customer_coverage["completed_sales_amount"]
    / customer_coverage["completed_sales_amount"].sum()
    * 100
).round(2)

customer_coverage

,customer_id_status,transaction_lines,completed_sales_amount,completed_sales_share_percent
0,Known Customer ID,406829,8911407.90,83.54
1,Missing Customer ID,135080,1755276.64,16.46


## Top Countries by Completed Sales

This section reviews the top countries by completed sales amount.

In [44]:
top_countries

,country,completed_transaction_lines,total_completed_sales
0,United Kingdom,485123,9025222.08
1,Netherlands,2359,285446.34
2,EIRE,7890,283453.96
3,Germany,9040,228867.14
4,France,8407,209715.11
5,Australia,1182,138521.31
6,Spain,2484,61577.11
7,Switzerland,1966,57089.90
8,Belgium,2031,41196.34
9,Sweden,451,38378.33


In [45]:
top_country = top_countries.iloc[0]

print(
    f"The top country by completed sales is {top_country['country']} "
    f"with ${top_country['total_completed_sales']:,.2f} in completed sales."
)

The top country by completed sales is United Kingdom with $9,025,222.08 in completed sales.


## Top Products by Completed Sales

This section reviews the highest-performing products by completed sales amount.

In [46]:
top_products

,stock_code,product_description,completed_transaction_lines,units_sold,total_completed_sales
0,DOT,DOTCOM POSTAGE,706,706,206248.77
1,22423,REGENCY CAKESTAND 3 TIER,2017,13879,174484.74
2,23843,"PAPER CRAFT , LITTLE BIRDIE",1,80995,168469.60
3,85123A,WHITE HANGING HEART T-LIGHT HOLDER,2265,37660,104518.80
4,47566,PARTY BUNTING,1706,18295,99504.33
5,85099B,JUMBO BAG RED RETROSPOT,2112,48474,94340.05
6,23166,MEDIUM CERAMIC TOP STORAGE JAR,250,78033,81700.92
7,M,Manual,321,7224,78110.27
8,POST,POSTAGE,1126,3150,78101.88
9,23084,RABBIT NIGHT LIGHT,1036,30788,66964.99


In [47]:
top_product = top_products.iloc[0]

print(
    f"The top product by completed sales is stock code {top_product['stock_code']} "
    f"with ${top_product['total_completed_sales']:,.2f} in completed sales."
)

The top product by completed sales is stock code DOT with $206,248.77 in completed sales.
